# LeetCode #48 — Rotate Image: xoay ma trận 90° tại chỗ

**Mức độ:** Medium · **Chủ đề:** Array, Math, Matrix

Skeleton trên LeetCode:

```python
class Solution:
    def rotate(self, matrix: List[List[int]]) -> None:
        """
        Do not return matrix, modify matrix in-place instead.
        """
```

Mục tiêu của notebook:

1. Tìm **công thức vị trí**: ô `(i, j)` đi đâu khi xoay 90° theo chiều kim đồng hồ.
2. Cách dễ nghĩ: tạo ma trận mới — và vì sao đề cấm.
3. Mẹo đẹp: **xoay = chuyển vị (transpose) + lật từng hàng**.
4. Cách thứ hai: xoay **từng lớp**, mỗi lần đổi chỗ **4 ô** theo vòng.

# 1. Bài toán là gì?

Cho ma trận vuông `n × n` biểu diễn một ảnh. Xoay ảnh **90° theo chiều kim đồng hồ**, **tại chỗ** (sửa trực tiếp `matrix`, **không** tạo ma trận 2 chiều khác).

**Constraints (theo đề):**

- `n == matrix.length == matrix[i].length`
- `1 <= n <= 20`
- `-1000 <= matrix[i][j] <= 1000`

---

## Ví dụ 1

```text
1 2 3        7 4 1
4 5 6   ->   8 5 2
7 8 9        9 6 3
```

## Ví dụ 2

```text
 5  1  9 11        15 13  2  5
 2  4  8 10   ->   14  3  4  1
13  3  6  7        12  6  8  9
15 14 12 16        16  7 10 11
```

Quan sát: **cột đầu tiên** (đọc từ dưới lên: 7, 4, 1) trở thành **hàng đầu tiên**.

In [1]:
from typing import List

def show(m):
    width = max(len(str(x)) for row in m for x in row)
    for row in m:
        print(" ".join(str(x).rjust(width) for x in row))

show([[1, 2, 3], [4, 5, 6], [7, 8, 9]])

1 2 3
4 5 6
7 8 9


# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Ma trận (mảng hai chiều) trong Python

Cuối mục có phần **2.2 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Ma trận (mảng hai chiều) trong Python

### Tạo và truy cập

Ma trận `m × n` là list gồm `m` hàng, mỗi hàng là list `n` phần tử. Ô hàng `i`, cột `j` là `M[i][j]`; số hàng `len(M)`, số cột `len(M[0])`.

```text
        j=0 j=1 j=2
i=0   [  1,  2,  3 ]
i=1   [  4,  5,  6 ]
```

In [2]:
M_mx = [[1, 2, 3],
        [4, 5, 6]]
print("M[1][2] =", M_mx[1][2], "| số hàng:", len(M_mx), "| số cột:", len(M_mx[0]))

dung_mx = [[0] * 3 for _ in range(2)]    # ĐÚNG: mỗi hàng là list riêng
sai_mx = [[0] * 3] * 2                   # SAI: 2 hàng là cùng một list
dung_mx[0][0] = sai_mx[0][0] = 7
print("tạo bằng for:", dung_mx, "| nhân list:", sai_mx)

M[1][2] = 6 | số hàng: 2 | số cột: 3
tạo bằng for: [[7, 0, 0], [0, 0, 0]] | nhân list: [[7, 0, 0], [7, 0, 0]]


### Duyệt và các phép biến đổi cơ bản

- Duyệt theo hàng: `for i in range(m): for j in range(n)`.
- **Chuyển vị** (transpose): `T[j][i] = M[i][j]` — hàng thành cột. Trong Python: `list(zip(*M))`.
- **Lật ngang** một hàng: `row[::-1]`; lật dọc: đảo thứ tự hàng `M[::-1]`.
- Xoay 90° theo chiều kim đồng hồ = chuyển vị rồi lật từng hàng.

In [3]:
def in_mt(A):
    for row in A:
        print("   ", " ".join(f"{v:>2}" for v in row))

A_mx = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
print("gốc:"); in_mt(A_mx)
T_mx = [list(r) for r in zip(*A_mx)]
print("chuyển vị:"); in_mt(T_mx)
print("xoay 90° phải (chuyển vị + lật hàng):"); in_mt([r[::-1] for r in T_mx])

gốc:
     1  2  3
     4  5  6
     7  8  9
chuyển vị:
     1  4  7
     2  5  8
     3  6  9
xoay 90° phải (chuyển vị + lật hàng):
     7  4  1
     8  5  2
     9  6  3


### Ô kề và mảng hướng

Bốn ô kề của `(r, c)`: lên `(r−1, c)`, xuống `(r+1, c)`, trái `(r, c−1)`, phải `(r, c+1)`. Viết thành **mảng hướng** để khỏi lặp code, và luôn **kiểm tra biên** `0 <= nr < m and 0 <= nc < n`.

In [4]:
HUONG = [(-1, 0), (1, 0), (0, -1), (0, 1)]       # lên, xuống, trái, phải
def o_ke(r, c, m, n):
    return [(r + dr, c + dc) for dr, dc in HUONG if 0 <= r + dr < m and 0 <= c + dc < n]
print("ô kề của (0, 0) trong lưới 3×3:", o_ke(0, 0, 3, 3))
print("ô kề của (1, 1):", o_ke(1, 1, 3, 3))

ô kề của (0, 0) trong lưới 3×3: [(1, 0), (0, 1)]
ô kề của (1, 1): [(0, 1), (2, 1), (1, 0), (1, 2)]


### Đánh số ô thành một chỉ số và ngược lại

Ma trận `m × n` "trải phẳng" theo hàng: ô `(i, j)` ↔ chỉ số `k = i * n + j`; ngược lại `i, j = divmod(k, n)`. Hữu ích để tìm nhị phân trên ma trận hoặc lưu ô vào set.

In [5]:
n_cols = 4
for k in [0, 5, 11]:
    print(f"k = {k:>2} → (i, j) = {divmod(k, n_cols)} → k = {divmod(k, n_cols)[0] * n_cols + divmod(k, n_cols)[1]}")

k =  0 → (i, j) = (0, 0) → k = 0
k =  5 → (i, j) = (1, 1) → k = 5
k = 11 → (i, j) = (2, 3) → k = 11


Lỗi hay gặp: `[[0]*n]*m`; nhầm `m` (hàng) với `n` (cột) khi ma trận không vuông; quên kiểm tra biên; sửa ma trận trong lúc vẫn cần giá trị cũ để tính ô khác.

## 2.2 Áp dụng vào bài này

Xoay **ma trận** 90° tại chỗ = **chuyển vị** (đổi `M[i][j]` với `M[j][i]`) rồi **lật từng hàng**. Hoặc xoay từng "lớp" bốn ô một — cần nắm chắc cách đánh chỉ số hàng/cột.

# 3. Công thức vị trí

Xoay 90° theo chiều kim đồng hồ, ô ở **hàng `i`, cột `j`** đi tới **hàng `j`, cột `n - 1 - i`**:

```text
(i, j)  ->  (j, n - 1 - i)
```

Kiểm tra với số `1` ở `(0, 0)`, `n = 3`: đi tới `(0, 2)` — góc trên phải. Đúng!
Số `7` ở `(2, 0)`: đi tới `(0, 0)`. Đúng!

Giải thích trực giác: **hàng `i`** trở thành **cột `n - 1 - i`** (hàng trên cùng thành cột phải nhất); **cột `j`** trở thành **hàng `j`**.

In [6]:
n = 3
for i, j in [(0, 0), (2, 0), (1, 2), (1, 1)]:
    print(f"({i}, {j}) -> ({j}, {n - 1 - i})")

(0, 0) -> (0, 2)
(2, 0) -> (0, 0)
(1, 2) -> (2, 1)
(1, 1) -> (1, 1)


# 4. Cách dễ nghĩ: tạo ma trận mới

In [7]:
def rotate_copy(matrix: List[List[int]]) -> List[List[int]]:
    n = len(matrix)
    result = [[0] * n for _ in range(n)]
    for i in range(n):
        for j in range(n):
            result[j][n - 1 - i] = matrix[i][j]
    return result

show(rotate_copy([[1, 2, 3], [4, 5, 6], [7, 8, 9]]))

7 4 1
8 5 2
9 6 3


**Vấn đề:** đề yêu cầu **tại chỗ**, không được cấp phát ma trận khác (bộ nhớ phụ phải `O(1)`).

Thử "tại chỗ" ngây thơ: viết thẳng `matrix[j][n-1-i] = matrix[i][j]` → **ghi đè** những ô chưa kịp đọc!

In [8]:
def wrong_naive_inplace(matrix: List[List[int]]) -> None:
    n = len(matrix)
    for i in range(n):
        for j in range(n):
            matrix[j][n - 1 - i] = matrix[i][j]      # ghi đè dữ liệu chưa đọc

m = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
wrong_naive_inplace(m)
show(m)

7 4 1
2 5 2
1 2 1


# 5. Mẹo: Xoay = Chuyển vị + Lật hàng

Tách công thức `(i, j) → (j, n-1-i)` thành **hai bước** đơn giản, mỗi bước dễ làm tại chỗ:

1. **Chuyển vị (transpose):** `(i, j) → (j, i)` — đổi hàng thành cột, lật qua đường chéo chính.
2. **Lật mỗi hàng trái–phải:** `(r, c) → (r, n-1-c)`.

Ghép lại: `(i, j) → (j, i) → (j, n-1-i)`. Đúng công thức!

```text
1 2 3    chuyển vị    1 4 7    lật hàng    7 4 1
4 5 6   ---------->   2 5 8   --------->   8 5 2
7 8 9                 3 6 9                9 6 3
```

## Chuyển vị tại chỗ

Chỉ đổi chỗ các cặp **phía trên đường chéo** với phía dưới (`j > i`). Nếu duyệt mọi `(i, j)`, mỗi cặp bị đổi **hai lần** → về như cũ.

```python
for i in range(n):
    for j in range(i + 1, n):
        matrix[i][j], matrix[j][i] = matrix[j][i], matrix[i][j]
```

## Lật hàng tại chỗ

`row.reverse()` đảo list tại chỗ.

In [9]:
def transpose(matrix: List[List[int]]) -> None:
    n = len(matrix)
    for i in range(n):
        for j in range(i + 1, n):
            matrix[i][j], matrix[j][i] = matrix[j][i], matrix[i][j]


def rotate(matrix: List[List[int]]) -> None:
    transpose(matrix)
    for row in matrix:
        row.reverse()


m = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
transpose(m)
print("Sau chuyển vị:")
show(m)
for row in m:
    row.reverse()
print("Sau lật hàng:")
show(m)

Sau chuyển vị:
1 4 7
2 5 8
3 6 9
Sau lật hàng:
7 4 1
8 5 2
9 6 3


# 6. Cách 2: xoay từng lớp, đổi chỗ 4 ô

Ma trận gồm các **lớp** đồng tâm (vỏ ngoài, vỏ trong, …). Khi xoay, mỗi ô chỉ di chuyển **trong lớp của nó**, và cứ **4 ô** tạo thành một vòng:

```text
trên  (i, j)          -> phải (j, n-1-i)
phải  (j, n-1-i)      -> dưới (n-1-i, n-1-j)
dưới  (n-1-i, n-1-j)  -> trái (n-1-j, i)
trái  (n-1-j, i)      -> trên (i, j)
```

Đổi chỗ 4 ô cùng lúc bằng một biến tạm (Python cho phép gán đồng thời 4 giá trị).

Duyệt `i` từ `0` tới `n // 2 - 1` (các lớp), `j` từ `i` tới `n - 2 - i` (các ô trên cạnh trên của lớp, trừ góc cuối — vì góc cuối thuộc vòng của góc đầu).

In [10]:
def rotate_layers(matrix: List[List[int]]) -> None:
    n = len(matrix)
    for i in range(n // 2):
        for j in range(i, n - 1 - i):
            top = matrix[i][j]
            matrix[i][j] = matrix[n - 1 - j][i]                  # trái -> trên
            matrix[n - 1 - j][i] = matrix[n - 1 - i][n - 1 - j]  # dưới -> trái
            matrix[n - 1 - i][n - 1 - j] = matrix[j][n - 1 - i]  # phải -> dưới
            matrix[j][n - 1 - i] = top                           # trên -> phải

m = [[5, 1, 9, 11], [2, 4, 8, 10], [13, 3, 6, 7], [15, 14, 12, 16]]
rotate_layers(m)
show(m)

15 13  2  5
14  3  4  1
12  6  8  9
16  7 10 11


# 7. Mô phỏng vòng 4 ô

In [11]:
def trace_rotate_layers(matrix: List[List[int]]) -> None:
    n = len(matrix)
    for i in range(n // 2):
        print(f"--- Lớp {i} ---")
        for j in range(i, n - 1 - i):
            cells = [(i, j), (j, n - 1 - i), (n - 1 - i, n - 1 - j), (n - 1 - j, i)]
            vals = [matrix[r][c] for r, c in cells]
            print(f"  vòng: trên{cells[0]}={vals[0]} -> phải{cells[1]}={vals[1]} -> dưới{cells[2]}={vals[2]} -> trái{cells[3]}={vals[3]} -> trên")
            top = matrix[i][j]
            matrix[i][j] = matrix[n - 1 - j][i]
            matrix[n - 1 - j][i] = matrix[n - 1 - i][n - 1 - j]
            matrix[n - 1 - i][n - 1 - j] = matrix[j][n - 1 - i]
            matrix[j][n - 1 - i] = top
    print("Kết quả:")
    show(matrix)

trace_rotate_layers([[5, 1, 9, 11], [2, 4, 8, 10], [13, 3, 6, 7], [15, 14, 12, 16]])

--- Lớp 0 ---
  vòng: trên(0, 0)=5 -> phải(0, 3)=11 -> dưới(3, 3)=16 -> trái(3, 0)=15 -> trên
  vòng: trên(0, 1)=1 -> phải(1, 3)=10 -> dưới(3, 2)=12 -> trái(2, 0)=13 -> trên
  vòng: trên(0, 2)=9 -> phải(2, 3)=7 -> dưới(3, 1)=14 -> trái(1, 0)=2 -> trên
--- Lớp 1 ---
  vòng: trên(1, 1)=4 -> phải(1, 2)=8 -> dưới(2, 2)=6 -> trái(2, 1)=3 -> trên
Kết quả:
15 13  2  5
14  3  4  1
12  6  8  9
16  7 10 11


Ma trận `4 × 4` có 2 lớp: lớp ngoài 3 vòng (12 ô), lớp trong 1 vòng (4 ô). Ma trận lẻ (`n = 3`) có ô **trung tâm** không di chuyển.

# 8. Phiên bản đúng format LeetCode

In [12]:
class Solution:
    def rotate(self, matrix: List[List[int]]) -> None:
        """
        Do not return matrix, modify matrix in-place instead.
        """
        n = len(matrix)
        for i in range(n):
            for j in range(i + 1, n):
                matrix[i][j], matrix[j][i] = matrix[j][i], matrix[i][j]
        for row in matrix:
            row.reverse()

In [13]:
def check(name, matrix, expected):
    original_rows = [id(r) for r in matrix]
    result = Solution().rotate(matrix)
    assert result is None
    assert matrix == expected, (name, matrix, expected)
    assert [id(r) for r in matrix] == original_rows, "phải sửa tại chỗ (các hàng vẫn là list cũ)"
    print(f"{name:<10} PASS")


check("Ví dụ 1", [[1, 2, 3], [4, 5, 6], [7, 8, 9]], [[7, 4, 1], [8, 5, 2], [9, 6, 3]])
check("Ví dụ 2", [[5, 1, 9, 11], [2, 4, 8, 10], [13, 3, 6, 7], [15, 14, 12, 16]],
      [[15, 13, 2, 5], [14, 3, 4, 1], [12, 6, 8, 9], [16, 7, 10, 11]])
check("1 x 1", [[42]], [[42]])
check("2 x 2", [[1, 2], [3, 4]], [[3, 1], [4, 2]])

import random
random.seed(48)
for _ in range(500):
    n = random.randint(1, 8)
    m = [[random.randint(-1000, 1000) for _ in range(n)] for _ in range(n)]
    expected = rotate_copy(m)
    a = [row[:] for row in m]; Solution().rotate(a)
    b = [row[:] for row in m]; rotate_layers(b)
    assert a == expected and b == expected
    c = [row[:] for row in m]
    for _ in range(4):
        rotate(c)
    assert c == m                          # xoay 4 lần quay về ban đầu
print("500 ma trận ngẫu nhiên: hai cách khớp công thức; xoay 4 lần quay về như cũ")

Ví dụ 1    PASS
Ví dụ 2    PASS
1 x 1      PASS
2 x 2      PASS
500 ma trận ngẫu nhiên: hai cách khớp công thức; xoay 4 lần quay về như cũ


# 9. Vì sao đúng? Độ phức tạp?

**Chuyển vị + lật:** đã chứng minh bằng công thức ở mục 5 — hợp của hai phép biến đổi vị trí đúng bằng phép xoay.

**Xoay lớp:** mỗi vòng 4 ô đúng là quỹ đạo của một ô dưới phép xoay (xoay 4 lần quay về chỗ cũ). Các vòng rời nhau và phủ toàn bộ ma trận (trừ ô trung tâm khi `n` lẻ), nên đổi chỗ từng vòng là đủ.

```text
Time:  O(n²)   — mỗi ô được di chuyển hằng số lần
Space: O(1)    — chỉ vài biến tạm
```

# 10. Những lỗi dễ mắc

## Lỗi 1: Chuyển vị bằng cách duyệt mọi `(i, j)`

Mỗi cặp bị đổi hai lần → ma trận không đổi.

In [14]:
def wrong_transpose(matrix: List[List[int]]) -> None:
    n = len(matrix)
    for i in range(n):
        for j in range(n):                       # SAI: phải là range(i + 1, n)
            matrix[i][j], matrix[j][i] = matrix[j][i], matrix[i][j]

m = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
wrong_transpose(m)
show(m)

1 2 3
4 5 6
7 8 9


## Lỗi 2: Lật sai chiều

Chuyển vị rồi **lật cột** (đảo thứ tự các hàng) sẽ xoay **ngược** chiều kim đồng hồ:

In [15]:
m = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
transpose(m)
m.reverse()                                       # đảo thứ tự hàng thay vì đảo từng hàng
show(m)

3 6 9
2 5 8
1 4 7


## Lỗi 3: Gán lại `matrix = ...` bên trong hàm

`matrix = rotate_copy(matrix)` chỉ đổi biến cục bộ; ma trận của LeetCode không thay đổi. Nếu buộc phải dùng bản sao (không khuyến khích), hãy dùng `matrix[:] = ...`.

## Lỗi 4: Tạo ma trận bằng `[[0] * n] * n`

Mọi hàng là **cùng một** list (cái bẫy đã gặp ở #6, #36).

# 11. Mẹo nhận diện pattern

**Biến đổi hình học trên ma trận = ghép các phép đơn giản:**

| Phép | Cách làm |
|---|---|
| Xoay 90° thuận chiều | chuyển vị + lật từng hàng |
| Xoay 90° ngược chiều | chuyển vị + đảo thứ tự hàng (hoặc lật hàng rồi chuyển vị) |
| Xoay 180° | lật từng hàng + đảo thứ tự hàng |
| Lật ngang / dọc | `row.reverse()` / `matrix.reverse()` |

Luôn bắt đầu bằng việc viết **công thức vị trí** `(i, j) → (?, ?)`, rồi tìm cách tách thành các phép dễ làm tại chỗ.

Bài liên quan: **#54 Spiral Matrix** (duyệt theo lớp), **#867 Transpose Matrix**, **#1886 Determine Whether Matrix Can Be Obtained By Rotation**.

# 12. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Xoay ma trận sau 90° thuận chiều bằng cách chuyển vị rồi lật hàng, ghi kết quả từng bước:

```text
1 2
3 4
5 6   (không vuông!)
```

Tại sao bài này không áp dụng được cho ma trận không vuông **tại chỗ**?

## Bài 2 — Chuyển vị tại chỗ

In [16]:
def exercise_2_transpose(matrix: List[List[int]]) -> None:
    # TODO: chỉ đổi chỗ các cặp j > i
    pass


# m = [[1, 2], [3, 4]]; exercise_2_transpose(m); assert m == [[1, 3], [2, 4]]
# m = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]; exercise_2_transpose(m); assert m == [[1, 4, 7], [2, 5, 8], [3, 6, 9]]

## Bài 3 — Xoay ngược chiều kim đồng hồ

Xoay 90° **ngược** chiều kim đồng hồ, tại chỗ.

```text
1 2 3        3 6 9
4 5 6   ->   2 5 8
7 8 9        1 4 7
```

In [17]:
def exercise_3_rotate_ccw(matrix: List[List[int]]) -> None:
    # TODO
    pass


# m = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]; exercise_3_rotate_ccw(m); assert m == [[3, 6, 9], [2, 5, 8], [1, 4, 7]]

## Bài 4 — Ma trận không vuông (tạo mới)

Xoay 90° thuận chiều một ma trận `m × n` bất kỳ, **trả về ma trận mới** `n × m`.

```python
rotate_rect([[1, 2], [3, 4], [5, 6]]) -> [[5, 3, 1], [6, 4, 2]]
```

In [18]:
def exercise_4_rotate_rect(matrix: List[List[int]]) -> List[List[int]]:
    # TODO
    pass


# assert exercise_4_rotate_rect([[1, 2], [3, 4], [5, 6]]) == [[5, 3, 1], [6, 4, 2]]
# assert exercise_4_rotate_rect([[1, 2, 3]]) == [[1], [2], [3]]

## Bài 5 — Mini challenge: kiểm tra có thể xoay ra (LeetCode #1886)

Cho hai ma trận vuông `mat` và `target`. Trả về `True` nếu xoay `mat` (0, 90, 180 hoặc 270 độ) được `target`.

```python
find_rotation([[0, 1], [1, 0]], [[1, 0], [0, 1]]) -> True
find_rotation([[0, 1], [1, 1]], [[1, 0], [0, 1]]) -> False
```

In [19]:
def exercise_5_find_rotation(mat: List[List[int]], target: List[List[int]]) -> bool:
    # TODO
    pass


# assert exercise_5_find_rotation([[0, 1], [1, 0]], [[1, 0], [0, 1]]) is True
# assert exercise_5_find_rotation([[0, 1], [1, 1]], [[1, 0], [0, 1]]) is False

# 13. Gợi ý

**Bài 1**
- Mức 1: chuyển vị ma trận `3 × 2` cho ra ma trận `2 × 3` — **kích thước thay đổi**, không thể chứa trong cùng các list cũ.

**Bài 2**
- Mức 1: `for j in range(i + 1, n)`.

**Bài 3**
- Mức 1: công thức ngược chiều: `(i, j) → (n-1-j, i)`.
- Mức 2: chuyển vị rồi `matrix.reverse()` (đảo thứ tự hàng).

**Bài 4**
- Mức 1: kết quả có `n` hàng, `m` cột; `result[j][m - 1 - i] = matrix[i][j]`.
- Mức 2: cách gọn: `[list(row) for row in zip(*matrix[::-1])]`.

**Bài 5**
- Mức 1: thử tối đa 4 lần: so sánh, rồi xoay `mat` thêm 90°.

# 14. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

Chuyển vị `[[1, 2], [3, 4], [5, 6]]` → `[[1, 3, 5], [2, 4, 6]]`, lật hàng → `[[5, 3, 1], [6, 4, 2]]`. Ma trận kết quả có **2 hàng × 3 cột** thay vì 3 × 2 — không thể "tại chỗ" trong các list có độ dài cố định. Vì vậy đề chỉ cho ma trận vuông.

## Đáp án Bài 2

In [20]:
def answer_2_transpose(matrix: List[List[int]]) -> None:
    n = len(matrix)
    for i in range(n):
        for j in range(i + 1, n):
            matrix[i][j], matrix[j][i] = matrix[j][i], matrix[i][j]

m = [[1, 2], [3, 4]]; answer_2_transpose(m); assert m == [[1, 3], [2, 4]]
m = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]; answer_2_transpose(m); assert m == [[1, 4, 7], [2, 5, 8], [3, 6, 9]]
print("Bài 2 PASS")

Bài 2 PASS


## Đáp án Bài 3

In [21]:
def answer_3_rotate_ccw(matrix: List[List[int]]) -> None:
    answer_2_transpose(matrix)
    matrix.reverse()

m = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]; answer_3_rotate_ccw(m); assert m == [[3, 6, 9], [2, 5, 8], [1, 4, 7]]
for _ in range(200):
    n = random.randint(1, 6)
    m0 = [[random.randint(0, 9) for _ in range(n)] for _ in range(n)]
    a = [r[:] for r in m0]
    answer_3_rotate_ccw(a)
    rotate(a)                     # ngược rồi thuận -> về như cũ
    assert a == m0
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [22]:
def answer_4_rotate_rect(matrix: List[List[int]]) -> List[List[int]]:
    m, n = len(matrix), len(matrix[0])
    result = [[0] * m for _ in range(n)]
    for i in range(m):
        for j in range(n):
            result[j][m - 1 - i] = matrix[i][j]
    return result

assert answer_4_rotate_rect([[1, 2], [3, 4], [5, 6]]) == [[5, 3, 1], [6, 4, 2]]
assert answer_4_rotate_rect([[1, 2, 3]]) == [[1], [2], [3]]
assert answer_4_rotate_rect([[1, 2], [3, 4], [5, 6]]) == [list(r) for r in zip(*[[1, 2], [3, 4], [5, 6]][::-1])]
print("Bài 4 PASS")

Bài 4 PASS


`zip(*matrix[::-1])`: đảo thứ tự hàng rồi "zip" các hàng lại theo cột — một cách viết xoay 90° rất gọn trong Python.

## Đáp án Bài 5

In [23]:
def answer_5_find_rotation(mat: List[List[int]], target: List[List[int]]) -> bool:
    cur = [row[:] for row in mat]
    for _ in range(4):
        if cur == target:
            return True
        rotate(cur)
    return False

assert answer_5_find_rotation([[0, 1], [1, 0]], [[1, 0], [0, 1]]) is True
assert answer_5_find_rotation([[0, 1], [1, 1]], [[1, 0], [0, 1]]) is False
print("Bài 5 PASS")

Bài 5 PASS


# 15. Tổng kết

> Xoay 90° thuận chiều: `(i, j) → (j, n-1-i)` = **chuyển vị** (`(i, j) → (j, i)`) rồi **lật mỗi hàng**. Cả hai bước đều làm tại chỗ dễ dàng.

---

## Checklist trước khi nộp

- [ ] Chuyển vị chỉ với `j > i`?
- [ ] Lật **từng hàng** (`row.reverse()`), không phải đảo thứ tự hàng?
- [ ] Sửa tại chỗ, không gán lại `matrix = ...`?
- [ ] Đã thử `1 × 1`, `2 × 2`, `n` lẻ và chẵn?